In [ ]:
dbutils.secrets.listScopes()


In [0]:
dbutils.secrets.list("secretscope")

[SecretMetadata(key='engdados-secret')]

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

name,url,comment
dbc_engdados,abfss://unity-catalog-storage@dbstoragentgl7y3kq4vea.dfs.core.windows.net/7405605509812525,null
ext_stoengdados_gold,abfss://lakehouse@stoengdados.dfs.core.windows.net/gold,null


In [0]:
scope="secretscope"
key="engdados-secret"
storage_account="stoengdados"
application_id = dbutils.secrets.get(scope, "application-id")
directory_id = dbutils.secrets.get(scope, "tenant-id")
service_credential = dbutils.secrets.get(scope=scope,key=key)

spark.conf.set("fs.azure.account.auth.type.%s.dfs.core.windows.net"%(storage_account), "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.%s.dfs.core.windows.net"%(storage_account), "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.%s.dfs.core.windows.net"%(storage_account), application_id)
spark.conf.set("fs.azure.account.oauth2.client.secret.%s.dfs.core.windows.net"%(storage_account), service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.%s.dfs.core.windows.net"%(storage_account), "https://login.microsoftonline.com/%s/oauth2/token"%(directory_id))



In [0]:
container_name = "lakehouse"
dbutils.fs.ls("abfss://%s@%s.dfs.core.windows.net/"%(container_name,storage_account))


[FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/bronze/', name='bronze/', size=0, modificationTime=1774401021000),
 FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/gold/', name='gold/', size=0, modificationTime=1774401041000),
 FileInfo(path='abfss://lakehouse@stoengdados.dfs.core.windows.net/silver/', name='silver/', size=0, modificationTime=1774401032000)]

In [0]:
%sql
    
USE CATALOG dbc_engdados;

CREATE SCHEMA IF NOT EXISTS gold
MANAGED LOCATION 'abfss://lakehouse@stoengdados.dfs.core.windows.net/gold';